# Robot Synthetic Data Generation Workshop

End-to-end pipeline: **Synthetic Data Generation → VLA Training → Simulation Evaluation**

| Step | Script | Description |
|------|--------|-------------|
| 0 | Setup | Install deps, check GPU, download scene assets |
| 1 | `01_gen_data.py` / `02_gen_data_custom_scene.py` | Generate Franka pick-cube trajectories |
| 2 | `02_train_vla.py` | Fine-tune SmolVLA on collected data |
| 3 | `03_eval.py` / `04_eval_custom_scene.py` | Closed-loop simulation evaluation |

---
## 0. Environment Setup

In [1]:
import os, sys, json, subprocess, time
from pathlib import Path
from IPython.display import display, Image, HTML, Video
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

os.environ['HF_HUB_OFFLINE'] = '1'

WORKSHOP_DIR = Path(os.getcwd())
SCRIPTS_DIR = WORKSHOP_DIR / 'scripts'
OUTPUT_DIR = Path('/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Workshop dir: {WORKSHOP_DIR}')
print(f'Output dir:   {OUTPUT_DIR}')
print(f'Python:       {sys.executable}')

Workshop dir: /workspace/workshop
Output dir:   /output
Python:       /opt/conda/envs/py_3.12/bin/python3.12


### 0.1 GPU & ROCm Check

In [2]:
import torch

print(f'PyTorch:  {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / 1024**3
        print(f'  GPU[{i}]: {props.name}  VRAM: {vram_gb:.1f} GB')
else:
    print('WARNING: No GPU detected.')

!rocm-smi --showuse 2>/dev/null | head -20 || echo 'rocm-smi not available'

PyTorch:  2.6.0+gitdbfe118
CUDA available: True
  GPU[0]: AMD Instinct MI308X  VRAM: 192.0 GB
  GPU[1]: AMD Instinct MI308X  VRAM: 192.0 GB
  GPU[2]: AMD Instinct MI308X  VRAM: 192.0 GB
  GPU[3]: AMD Instinct MI308X  VRAM: 192.0 GB
  GPU[4]: AMD Instinct MI308X  VRAM: 192.0 GB
  GPU[5]: AMD Instinct MI308X  VRAM: 192.0 GB
  GPU[6]: AMD Instinct MI308X  VRAM: 192.0 GB
  GPU[7]: AMD Instinct MI308X  VRAM: 192.0 GB




============================ ROCm System Management Interface ============================
=================================== % time GPU is busy ===================================
GPU[0]		: GPU use (%): 0
GPU[1]		: GPU use (%): 0
GPU[2]		: GPU use (%): 0
GPU[3]		: GPU use (%): 0
GPU[4]		: GPU use (%): 0
GPU[5]		: GPU use (%): 0
GPU[6]		: GPU use (%): 0
GPU[7]		: GPU use (%): 0
================================== End of ROCm SMI Log ===================================


### 0.2 Install Dependencies

In [3]:
!pip install -q lerobot genesis-world transformers accelerate safetensors matplotlib Pillow 2>&1 | tail -5

In [4]:
import lerobot
print(f'lerobot: {lerobot.__version__}')

try:
    import genesis as gs
    print(f'genesis: {gs.__version__}')
except Exception as e:
    print(f'genesis import: {e}')

lerobot: 0.5.0


[I 04/07/26 11:26:35.118 1292] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout
/opt/conda/envs/py_3.12/lib/python3.12/site-packages/genesis/__init__.py:39: UserWarning: 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
  warn("'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/")


genesis: 0.4.5


### 0.3 Download Kitchen Scene Assets (for custom-scene steps)

In [5]:
!python {SCRIPTS_DIR}/00_download_kitchen.py --mesh-only

Asset directory: /workspace/workshop/assets/rustic_kitchen

--- Mesh assets (for Genesis simulation) ---
  [skip] rustic_kitchen_hq.glb already exists (126.9 MB)
  [skip] rustic_kitchen_collider.glb already exists (2.8 MB)

[skip] --mesh-only: skipping Gaussian Splat PLY and panorama

Done. Files ready for 01_inspect_scene.py / 03_pick_cube.py
  Mesh GLB -> Genesis (visual + collision)
  Splat PLY -> Isaac Sim (PLY -> USDZ via 3DGRUT) or Spark viewer


In [6]:
asset_dir = WORKSHOP_DIR / 'assets' / 'rustic_kitchen'
if asset_dir.exists():
    for f in sorted(asset_dir.iterdir()):
        size_mb = f.stat().st_size / 1024**2
        print(f'  {f.name:40s}  {size_mb:>8.1f} MB')
else:
    print(f'Asset dir not found: {asset_dir}')

  rustic_kitchen_collider.glb                    2.8 MB
  rustic_kitchen_hq.glb                        126.9 MB


---
## 1. Synthetic Data Generation

Generate Franka 7-DOF pick-cube expert trajectories using IK planning in Genesis.

- **Robot**: Franka Panda (9 DOF: 7 arm + 2 finger)
- **Trajectory**: HOME → hover → descend → grasp → lift
- **Cameras**: up + side (640×480)

### 1a. Default Flat Scene

In [7]:
N_EPISODES_FLAT = 10
REPO_ID_FLAT = 'local/workshop-flat'

!python {SCRIPTS_DIR}/01_gen_data.py \
    --n-episodes {N_EPISODES_FLAT} \
    --repo-id {REPO_ID_FLAT} \
    --save {OUTPUT_DIR} \
    --seed 42 \
    --no-videos \
    --task 'Pick up the red cube.'

[display] Xvfb started (PID=1540)


[I 04/07/26 11:26:50.221 1475] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/genesis/__init__.py:39: UserWarning: 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
  warn("'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/")


[Genesis] [11:26:52] [WARNING] 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/z3/z3core.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Traceback (most recent call last):
  File "/workspace/workshop/scripts/01_gen_data.py", line 400, in <module>
    main()
  File "/workspace/workshop/scripts/01_gen_data.py", line 129, in main
    scene = gs.Scene(
            ^^^^^^^^^
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/genesis/utils/misc.py", line 128, in new_init
    original_init(self, *args, **kwargs)
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/genesis/engine/scene.py", line 112, in __init__
    from genesis.engine.simulator import Simulator
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/genesis/engine/simulator.py", line 24, in <module>
    from .entities import HybridEntity
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/genesis/engine/entities/__init__.py", line 2, in <module>
    from .drone_entity import DroneEntity
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/genesis/engine/entities/drone_entity.py", line 9, in <module>
    from .rigid_entit

### 1b. Custom Kitchen Scene

In [8]:
N_EPISODES_KITCHEN = 10
REPO_ID_KITCHEN = 'local/workshop-kitchen'

!python {SCRIPTS_DIR}/02_gen_data_custom_scene.py \
    --n-episodes {N_EPISODES_KITCHEN} \
    --repo-id {REPO_ID_KITCHEN} \
    --save {OUTPUT_DIR} \
    --seed 42 \
    --no-videos \
    --task 'Pick up the red cube.'

[I 04/07/26 11:27:09.436 1662] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/genesis/__init__.py:39: UserWarning: 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
  warn("'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/")


[Genesis] [11:27:11] [WARNING] 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
[anchor] floor_origin: base_xy=(0.00, 0.00) base_lift=0.00 yaw=0° surface_z=-0.68
[anchor] Origin on floor, cube on floor, simplest pick setup
[placement] floor_z=-0.680 base_lift=0.000 -> base_z=-0.680


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/z3/z3core.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Traceback (most recent call last):
  File "/workspace/workshop/scripts/02_gen_data_custom_scene.py", line 379, in <module>
    main()
  File "/workspace/workshop/scripts/02_gen_data_custom_scene.py", line 107, in main
    scene, franka, cube, cam_ov, cam_front, cam_up, cam_side, info = build_scene(args, gs)
                                                                     ^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/workshop/scripts/pick_common.py", line 255, in build_scene
    scene = gs.Scene(
            ^^^^^^^^^
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/genesis/utils/misc.py", line 128, in new_init
    original_init(self, *args, **kwargs)
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/genesis/engine/scene.py", line 112, in __init__
    from genesis.engine.simulator import Simulator
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/genesis/engine/simulator.py", line 24, in <module>
    from .entities import HybridEntity
  File "/opt/conda/

### 1c. Visualize Generated Data

In [9]:
import numpy as np

try:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
except ImportError:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds_root = Path.home() / '.cache' / 'huggingface' / 'lerobot' / REPO_ID_FLAT
ds = LeRobotDataset(REPO_ID_FLAT, root=ds_root)
print(f'Dataset: {REPO_ID_FLAT}  (root: {ds_root})')
print(f'  Episodes:     {ds.num_episodes}')
print(f'  Total frames: {ds.total_frames}')
print(f'  FPS:          {ds.fps}')
print(f'  Features:     {list(ds.features.keys())}')

sample = ds[0]
print(f'\nSample keys: {list(sample.keys())}')
for k, v in sample.items():
    if hasattr(v, 'shape'):
        print(f'  {k}: shape={v.shape} dtype={v.dtype}')
    elif isinstance(v, str):
        print(f'  {k}: "{v}"')
    else:
        print(f'  {k}: {type(v).__name__} = {v}')

/opt/conda/envs/py_3.12/lib/python3.12/site-packages/redis/connection.py:77: UserWarning: redis-py works best with hiredis. Please consider installing
  warnings.warn(msg)


OfflineModeIsEnabled: Cannot reach https://huggingface.co/api/datasets/local/workshop-flat/refs: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.

In [10]:
# Visualize camera images from a sample episode
from PIL import Image as PILImage

def show_episode_frames(dataset, ep_idx=0, n_frames=6):
    if hasattr(dataset, 'episode_data_index'):
        start = dataset.episode_data_index['from'][ep_idx].item()
        end = dataset.episode_data_index['to'][ep_idx].item()
    else:
        fpe = dataset.total_frames // dataset.num_episodes
        start, end = ep_idx * fpe, (ep_idx + 1) * fpe

    indices = np.linspace(start, end - 1, n_frames, dtype=int)
    img_keys = [k for k in dataset.features if 'images' in k]
    if not img_keys:
        print('No image features found')
        return

    fig, axes = plt.subplots(len(img_keys), n_frames,
                             figsize=(3 * n_frames, 3 * len(img_keys)))
    if len(img_keys) == 1:
        axes = axes[np.newaxis, :]

    for row, key in enumerate(img_keys):
        for col, idx in enumerate(indices):
            sample = dataset[int(idx)]
            img = sample[key]
            if hasattr(img, 'numpy'):
                img = img.numpy()
            if img.ndim == 3 and img.shape[0] in (1, 3):
                img = np.transpose(img, (1, 2, 0))
            if img.dtype != np.uint8:
                if img.max() <= 1.0:
                    img = (img * 255).astype(np.uint8)
                else:
                    img = img.astype(np.uint8)
            axes[row, col].imshow(img)
            axes[row, col].set_title(f'frame {idx - start}', fontsize=8)
            axes[row, col].axis('off')
        axes[row, 0].set_ylabel(key.split('.')[-1], fontsize=10)

    fig.suptitle(f'Episode {ep_idx} Camera Views', fontsize=12)
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / f'ep{ep_idx}_camera_views.png'), bbox_inches='tight')
    plt.show()
    print(f'Saved: {OUTPUT_DIR / f"ep{ep_idx}_camera_views.png"}')

show_episode_frames(ds, ep_idx=0)

NameError: name 'ds' is not defined

In [11]:
# Plot joint trajectories for episode 0
def plot_joint_trajectory(dataset, ep_idx=0):
    if hasattr(dataset, 'episode_data_index'):
        start = dataset.episode_data_index['from'][ep_idx].item()
        end = dataset.episode_data_index['to'][ep_idx].item()
    else:
        fpe = dataset.total_frames // dataset.num_episodes
        start, end = ep_idx * fpe, (ep_idx + 1) * fpe

    states, actions = [], []
    for i in range(start, end):
        s = dataset[i]
        states.append(s['observation.state'].numpy())
        actions.append(s['action'].numpy())

    states = np.array(states)
    actions = np.array(actions)
    joint_names = ['j1','j2','j3','j4','j5','j6','j7','f1','f2']

    fig, axes = plt.subplots(3, 3, figsize=(14, 8), sharex=True)
    for i, ax in enumerate(axes.flat):
        if i >= states.shape[1]:
            ax.set_visible(False)
            continue
        ax.plot(states[:, i], label='state', linewidth=1)
        ax.plot(actions[:, i], label='action', linewidth=1, alpha=0.7)
        name = joint_names[i] if i < len(joint_names) else f'dim{i}'
        ax.set_title(name, fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    fig.suptitle(f'Episode {ep_idx} Joint Trajectories (state vs action)', fontsize=12)
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / f'ep{ep_idx}_joint_trajectory.png'), bbox_inches='tight')
    plt.show()

plot_joint_trajectory(ds, ep_idx=0)

NameError: name 'ds' is not defined

In [12]:
# Cube XY scatter: show all episode spawn positions + success/fail
def plot_cube_scatter(gen_dir):
    labels_path = gen_dir / 'episode_labels.json'
    summary_path = gen_dir / 'gen_summary.json'
    if not labels_path.exists():
        print(f'Labels not found: {labels_path}')
        return
    labels = json.loads(labels_path.read_text())
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}

    fig, ax = plt.subplots(figsize=(6, 5))
    for ep in labels:
        xy = ep.get('cube_xy') or ep.get('cube_world_xy') or ep.get('cube_local_dxdy')
        if xy is None:
            continue
        color = '#2ecc71' if ep['success'] else '#e74c3c'
        marker = 'o' if ep['success'] else 'x'
        ax.scatter(xy[0], xy[1], c=color, marker=marker, s=60, edgecolors='k', linewidths=0.5)

    ax.set_xlabel('X'); ax.set_ylabel('Y')
    sr = summary.get('success_rate', 0)
    ax.set_title(f'Cube Spawn Positions \u2014 SR: {sr:.0%}')
    ax.grid(True, alpha=0.3); ax.set_aspect('equal')
    plt.tight_layout()
    out = gen_dir / 'cube_xy_scatter.png'
    fig.savefig(str(out), bbox_inches='tight')
    plt.show()

plot_cube_scatter(OUTPUT_DIR / 'franka_gen_pick')

Labels not found: /output/franka_gen_pick/episode_labels.json


---
## 2. SmolVLA Post-Training

Fine-tune `lerobot/smolvla_base` (~450M params) on the generated dataset.
- Freeze vision encoder, train expert layers + state projection
- chunk_size=50, AdamW optimizer

In [13]:
N_TRAIN_STEPS = 200
BATCH_SIZE = 4
TRAIN_REPO = REPO_ID_FLAT
SAVE_DIR = OUTPUT_DIR / 'outputs' / 'workshop_smolvla'

!python {SCRIPTS_DIR}/02_train_vla.py \
    --dataset-id {TRAIN_REPO} \
    --n-steps {N_TRAIN_STEPS} \
    --batch-size {BATCH_SIZE} \
    --num-workers 0 \
    --save-dir {SAVE_DIR}

[train] device: cuda


  GPU[0]: AMD Instinct MI308X  VRAM: 192.0 GB
  GPU[1]: AMD Instinct MI308X  VRAM: 192.0 GB
  GPU[2]: AMD Instinct MI308X  VRAM: 192.0 GB
  GPU[3]: AMD Instinct MI308X  VRAM: 192.0 GB
  GPU[4]: AMD Instinct MI308X  VRAM: 192.0 GB
  GPU[5]: AMD Instinct MI308X  VRAM: 192.0 GB
  GPU[6]: AMD Instinct MI308X  VRAM: 192.0 GB
  GPU[7]: AMD Instinct MI308X  VRAM: 192.0 GB


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/redis/connection.py:77: UserWarning: redis-py works best with hiredis. Please consider installing
  warnings.warn(msg)


Traceback (most recent call last):
  File "/workspace/workshop/scripts/02_train_vla.py", line 125, in main
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
ModuleNotFoundError: No module named 'lerobot.common'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/workspace/workshop/scripts/02_train_vla.py", line 341, in <module>
    main()
  File "/workspace/workshop/scripts/02_train_vla.py", line 134, in main
    from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/lerobot/policies/__init__.py", line 17, in <module>
    from .groot.configuration_groot import GrootConfig as GrootConfig
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/lerobot/policies/groot/__init__.py", line 18, in <module>
    from .modeling_groot import GrootPolicy
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packa

### 2.1 Training Loss Curve

In [14]:
def plot_loss_curve(save_dir):
    metrics_path = save_dir / 'train_metrics.json'
    summary_path = save_dir / 'train_summary.json'
    if not metrics_path.exists():
        print(f'Metrics not found: {metrics_path}')
        return

    metrics = json.loads(metrics_path.read_text())
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}

    steps = [m['step'] for m in metrics]
    losses = [m['loss'] for m in metrics]
    grad_norms = [m['grad_norm'] for m in metrics]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(steps, losses, linewidth=0.8, alpha=0.5, label='per-step')
    window = max(1, len(losses) // 20)
    if len(losses) > window:
        smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
        ax1.plot(steps[window-1:], smoothed, linewidth=2, color='#e74c3c', label=f'smooth(w={window})')
    ax1.set_xlabel('Step'); ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(steps, grad_norms, linewidth=0.8, alpha=0.6, color='#3498db')
    ax2.set_xlabel('Step'); ax2.set_ylabel('Grad Norm')
    ax2.set_title('Gradient Norm'); ax2.grid(True, alpha=0.3)

    info = ''
    if summary:
        info = (f"loss: {summary.get('loss_start','?'):.4f} -> {summary.get('loss_end','?'):.4f}  "
                f"time: {summary.get('total_time_s',0):.0f}s  VRAM: {summary.get('peak_vram_mb',0):.0f}MB")
    fig.suptitle(f'SmolVLA Training  |  {info}', fontsize=10)
    plt.tight_layout()
    fig.savefig(str(save_dir / 'loss_curve.png'), bbox_inches='tight')
    plt.show()

plot_loss_curve(SAVE_DIR)

Metrics not found: /output/outputs/workshop_smolvla/train_metrics.json


In [15]:
summary_path = SAVE_DIR / 'train_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('=== Training Summary ===')
    for k, v in summary.items():
        if k not in ('selected_episodes',):
            print(f'  {k}: {v}')

---
## 3. Simulation Evaluation

Closed-loop: render cameras → SmolVLA inference → PD control → step physics.

### 3a. Eval on Unseen Positions (OOD)

In [16]:
CHECKPOINT = SAVE_DIR / 'final'
EVAL_UNSEEN_DIR = OUTPUT_DIR / 'eval_unseen'

!python {SCRIPTS_DIR}/03_eval.py \
    --policy-type smolvla \
    --checkpoint {CHECKPOINT} \
    --dataset-id {TRAIN_REPO} \
    --n-episodes 5 --max-steps 150 --seed 99 \
    --record-video \
    --save {EVAL_UNSEEN_DIR}

[I 04/07/26 11:27:48.517 1963] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/genesis/__init__.py:39: UserWarning: 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
  warn("'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/")


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/redis/connection.py:77: UserWarning: redis-py works best with hiredis. Please consider installing
  warnings.warn(msg)


Traceback (most recent call last):
  File "/workspace/workshop/scripts/03_eval.py", line 347, in main
    from lerobot.common.datasets.lerobot_dataset import LeRobotDatasetMetadata
ModuleNotFoundError: No module named 'lerobot.common'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/workspace/workshop/scripts/03_eval.py", line 681, in <module>
    main()
  File "/workspace/workshop/scripts/03_eval.py", line 356, in main
    from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/lerobot/policies/__init__.py", line 17, in <module>
    from .groot.configuration_groot import GrootConfig as GrootConfig
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/lerobot/policies/groot/__init__.py", line 18, in <module>
    from .modeling_groot import GrootPolicy
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/lerobot/policies/groot/mode

### 3b. Eval on Training Positions (IID)

In [17]:
EVAL_TRAIN_DIR = OUTPUT_DIR / 'eval_train'

!python {SCRIPTS_DIR}/03_eval.py \
    --policy-type smolvla \
    --checkpoint {CHECKPOINT} \
    --dataset-id {TRAIN_REPO} \
    --n-episodes 5 --max-steps 150 --seed 42 \
    --record-video \
    --save {EVAL_TRAIN_DIR}

[I 04/07/26 11:28:08.682 2106] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/genesis/__init__.py:39: UserWarning: 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
  warn("'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/")


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/redis/connection.py:77: UserWarning: redis-py works best with hiredis. Please consider installing
  warnings.warn(msg)


Traceback (most recent call last):
  File "/workspace/workshop/scripts/03_eval.py", line 347, in main
    from lerobot.common.datasets.lerobot_dataset import LeRobotDatasetMetadata
ModuleNotFoundError: No module named 'lerobot.common'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/workspace/workshop/scripts/03_eval.py", line 681, in <module>
    main()
  File "/workspace/workshop/scripts/03_eval.py", line 356, in main
    from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/lerobot/policies/__init__.py", line 17, in <module>
    from .groot.configuration_groot import GrootConfig as GrootConfig
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/lerobot/policies/groot/__init__.py", line 18, in <module>
    from .modeling_groot import GrootPolicy
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/lerobot/policies/groot/mode

### 3c. Visualize Evaluation Results

In [18]:
def show_eval_results(eval_dir, label=''):
    sp = eval_dir / 'eval_summary.json'
    if not sp.exists():
        print(f'No eval summary at {sp}')
        return None
    summary = json.loads(sp.read_text())
    results = summary.get('results', [])

    print(f'\n=== Eval: {label} ===')
    print(f"  Success: {summary['n_success']}/{summary['n_episodes']} = {summary['success_rate']:.0%}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    eps = [r['episode'] for r in results]
    lifts = [r['max_lift_m'] for r in results]
    colors = ['#2ecc71' if r['success'] else '#e74c3c' for r in results]
    axes[0].bar(eps, lifts, color=colors, edgecolor='k', linewidth=0.5)
    axes[0].axhline(y=0.02, color='orange', ls='--', label='2cm')
    axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Max Lift (m)')
    axes[0].set_title('Max Lift Height'); axes[0].legend(fontsize=8)

    sustains = [r['sustain_frames'] for r in results]
    axes[1].bar(eps, sustains, color=colors, edgecolor='k', linewidth=0.5)
    axes[1].axhline(y=8, color='orange', ls='--', label='8 frames')
    axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Sustain')
    axes[1].set_title('Lift Duration'); axes[1].legend(fontsize=8)

    for r in results:
        xy = r.get('cube_xy', r.get('cube_world_xy', [0,0]))
        c = '#2ecc71' if r['success'] else '#e74c3c'
        m = 'o' if r['success'] else 'x'
        axes[2].scatter(xy[0], xy[1], c=c, marker=m, s=80, edgecolors='k', linewidths=0.5)
    axes[2].set_xlabel('X'); axes[2].set_ylabel('Y')
    axes[2].set_title('Cube Positions'); axes[2].set_aspect('equal')
    axes[2].grid(True, alpha=0.3)

    fig.suptitle(f'{label}  |  {summary["n_success"]}/{summary["n_episodes"]}', fontsize=12)
    plt.tight_layout()
    fig.savefig(str(eval_dir / 'eval_visualization.png'), bbox_inches='tight')
    plt.show()
    return summary

s_unseen = show_eval_results(EVAL_UNSEEN_DIR, 'Unseen (seed=99)')
s_train = show_eval_results(EVAL_TRAIN_DIR, 'Training (seed=42)')

No eval summary at /output/eval_unseen/eval_summary.json


No eval summary at /output/eval_train/eval_summary.json


In [19]:
# Comparison bar chart
if s_unseen and s_train:
    fig, ax = plt.subplots(figsize=(6, 4))
    labels_bar = ['Unseen (OOD)', 'Training (IID)']
    rates = [s_unseen['success_rate'], s_train['success_rate']]
    bar_colors = ['#3498db', '#2ecc71']
    bars = ax.bar(labels_bar, rates, color=bar_colors, edgecolor='k', linewidth=0.5)
    for bar, rate in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{rate:.0%}', ha='center', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.15); ax.set_ylabel('Success Rate')
    ax.set_title('Evaluation Comparison'); ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / 'eval_comparison.png'), bbox_inches='tight')
    plt.show()

### 3d. Replay Evaluation Videos

In [20]:
def show_eval_videos(eval_dir, max_videos=4):
    vid_dir = eval_dir / 'videos'
    if not vid_dir.exists():
        print(f'No video dir: {vid_dir}')
        return
    videos = sorted(vid_dir.glob('*.mp4'))
    if not videos:
        print('No MP4 files found')
        return
    print(f'Found {len(videos)} videos')
    for vp in videos[:max_videos]:
        size_mb = vp.stat().st_size / 1024**2
        print(f'\n  {vp.name} ({size_mb:.1f} MB)')
        try:
            display(Video(str(vp), embed=True, width=480))
        except Exception as e:
            print(f'  Cannot embed video: {e}')
            try:
                frame_path = vp.with_suffix('.png')
                subprocess.run(['ffmpeg','-y','-i',str(vp),'-vframes','1',str(frame_path)],
                               capture_output=True)
                if frame_path.exists():
                    display(Image(filename=str(frame_path), width=480))
            except Exception:
                pass

show_eval_videos(EVAL_UNSEEN_DIR)

No video dir: /output/eval_unseen/videos


---
## 4. Summary & Artifacts

In [21]:
print('=== Generated Artifacts ===')
pngs = sorted(OUTPUT_DIR.rglob('*.png'))
mp4s = sorted(OUTPUT_DIR.rglob('*.mp4'))
jsons = sorted(OUTPUT_DIR.rglob('*.json'))

print(f'\nPNG images ({len(pngs)}):')
for p in pngs:
    print(f'  {p.relative_to(OUTPUT_DIR)}  ({p.stat().st_size/1024:.0f} KB)')

print(f'\nMP4 videos ({len(mp4s)}):')
for p in mp4s:
    print(f'  {p.relative_to(OUTPUT_DIR)}  ({p.stat().st_size/1024**2:.1f} MB)')

print(f'\nJSON summaries ({len(jsons)}):')
for p in jsons:
    print(f'  {p.relative_to(OUTPUT_DIR)}')

=== Generated Artifacts ===

PNG images (0):

MP4 videos (0):

JSON summaries (0):


In [22]:
# Display all generated PNGs inline
for p in pngs:
    print(f'\n--- {p.relative_to(OUTPUT_DIR)} ---')
    display(Image(filename=str(p), width=700))

In [23]:
print('\nWorkshop pipeline complete!')
print(f'All outputs saved to: {OUTPUT_DIR}')


Workshop pipeline complete!
All outputs saved to: /output
